# 图编译扩展能力：自定义融合 Pass

上一节我们学习了如何把自定义算子接入 GE 图。本节进入「图编译扩展能力」的第二部分：**自定义融合 Pass**——在图编译过程中识别图里一段符合规则的结构，并把它替换成另一段等价、但更高效的结构。

GE 内置了大量融合 Pass（如算子融合、UB 融合）。当内置规则不能覆盖你的业务场景时，可以编写自定义融合 Pass，由 GE 在图编译的指定阶段自动调用，对图做你期望的变换——同样**不需要修改 GE 框架代码**，Pass 以 `.so` 插件形式安装到指定目录被加载。

本节学习大纲如下：

- 融合 Pass 解决什么问题
- 两种实现方式：Pattern 匹配 vs 改图接口
- 一次 PatternFusionPass 的执行：匹配 → 决策 → 替换
- 用 PatternFusionPass 实现 MatMul+Add → GEMM（完整示例）
- Pattern 的边界规则（最易错点）
- 常用扩展点：CaptureTensor / PatternMatcherConfig / DecomposePass
- Pass 执行阶段（Stage）的选择
- 用改图接口实现 Pass（更细粒度控制）
- 编译部署与融合验证
- 进阶：自定义逻辑流分配 Pass 定制并发

## 1. 融合 Pass 解决什么问题

融合 Pass 的目标很直接——在图中找到一段符合规则的小结构，把它替换成另一段**等价**结构，从而减少算子数量、减少中间 Tensor 读写，或把图改成后端更容易高效执行的形态。这类变换不改变模型语义。

经典例子：图里有一段 `MatMul + Add`，若目标硬件上 `GEMM` 能一次完成同样的计算，就可以把它替换为单个 `GEMM`：

```
   a ----\                                  a ----\
          MatMul ----\                              \
   b ----/            Add ---- out   ==>    b ------- GEMM ---- out
   c ----------------/                              /
                                            c ----/
```

再比如 `Add(x, 0)`，输出与 `x` 等价，可直接替换为 `x`，消除一个无效算子。

GE 提供两类常用 Pattern 基类：

| 接口 | 适用场景 | 直观理解 |
| --- | --- | --- |
| `PatternFusionPass` | 匹配一段子图，替换成另一段子图（1:1 或复杂拓扑替换） | 「找到这个形状的局部图，整体替换」 |
| `DecomposePass` | 匹配某一种单个算子，替换成多个算子组成的子图 | 「把一个复杂算子拆成几个基础算子」 |

## 2. 两种实现方式

GE 支持两种通过 Pass 修改 Graph 的方式：

| 改图方式 | 基于 Pattern 匹配（推荐） | 基于改图接口 |
| --- | --- | --- |
| 核心思路 | 用 Pattern 定义匹配模板，框架自动完成匹配查找，开发者只需定义替换结构 | 直接编写改图函数，手动遍历图、查找节点、创建新节点、建边、删旧节点 |
| 关键特性 | 只需三步：匹配 → 决策 → 替换 | 需手动处理所有图修改细节，代码量大 |
| 适用场景 | 未触发约束即可使用，是首选方式 | 简单一次性修改、对图结构有完全控制需求 |
| 约束 | 不支持控制边匹配、不支持嵌套子图、不支持动态输入/输出个数节点 | 无上述约束，但需自己保证图的正确性 |

<p align="left"><img src="./images/fusion_pass_routes.svg" alt="自定义融合 Pass 两条实现路线" width="90%"></p>

> 建议：初次开发优先用 **Pattern 匹配**（路线 A），它把繁琐的匹配/重连交给框架；只有当 Pattern 无法表达你的变换（如涉及控制边、动态输入个数），或需要完全手动控制时，才用改图接口（路线 B）。

## 3. 一次 PatternFusionPass 如何执行

一次 `PatternFusionPass` 的执行可拆为五步，对应三个核心可重写函数：

```
定义 pattern → 在目标图中匹配 → 按条件过滤 → 生成 replacement → 替换并重连边
  Patterns()      (框架自动)      MeetRequirements()   Replacement()   (框架自动)
```

`PatternFusionPass` 类声明（节选）：

```cpp
class PatternFusionPass : public FusionBasePass {
 public:
  Status Run(GraphPtr &graph, CustomPassContext &pass_context) override;
 protected:
  virtual std::vector<PatternUniqPtr> Patterns() = 0;                              // 必须重写
  virtual bool MeetRequirements(const std::unique_ptr<MatchResult> &match_result);  // 可选，默认 true
  virtual GraphUniqPtr Replacement(const std::unique_ptr<MatchResult> &match_result) = 0;  // 必须重写
};
```

> **版本说明**：为兼容本教程的 CANN 9.0.0 环境，正文使用 `PatternFusionPass`（V1）。CANN 9.1.0 新增 `PatternFusionPassV2`，其 `MeetRequirements` 和 `Replacement` 会额外接收 `CustomPassContext`；V1 与 V2 均使用 `REG_FUSION_PASS` 注册。

| 函数 | 说明 | 是否必须重写 |
| --- | --- | --- |
| `Patterns` | 定义在目标图中匹配的模板拓扑，返回一个或多个图结构 | 是 |
| `MeetRequirements` | 对匹配到的结构按条件过滤，返回是否替换 | 否（默认返回 true） |
| `Replacement` | 定义替换结构，返回替换后的子图 | 是 |

涉及的核心概念：

- **Pattern（模式）**：描述子图结构特征的模板，匹配算法据此在 Graph 中查找符合规则的子图。
- **PatternMatcher（匹配器）**：执行匹配算法的核心对象，接收 Pattern 在 Graph 中查找子图。
- **GraphRewriter（重写器）**：执行改图，把匹配子图替换为 Replacement 结构并完成重连。
- **MatchResult（匹配结果）**：记录命中的真实节点、真实边以及主动捕获的 Tensor。

## 4. 完整示例：MatMul+Add → GEMM

下面用 `PatternFusionPass` 实现把图中 `MatMul + Add` 融合为单个 `GEMM`。这里参考 GE 仓 `examples/fusion_pass/pattern_base_pass/1_fuse_matmul_add_pass` 样例并做了简化：只保留 `MatMul+Add` pattern；原始样例还包含 `BatchMatMulV2+Add` pattern。

```
// a  b
// \ /                a    b    c
// MatMul    c   ==>   \   |   /
//    \     /            GEMM
//      Add
```

### 4.1 头文件

```cpp
#include <iostream>
#include "ge/fusion/pass/pattern_fusion_pass.h"  // 自定义融合 Pass 接口
#include "es_all_ops.h"                          // ES 接口（用于构建 pattern / replacement）

using namespace ge;
using namespace ge::fusion;
```

### 4.2 Pass 实现

```cpp
class FuseMatMulAndAddPass : public PatternFusionPass {
 protected:
  // 1) 定义匹配模板：MatMul(a,b) + Add(matmul, c)
  std::vector<PatternUniqPtr> Patterns() override {
    std::vector<PatternUniqPtr> patterns;
    auto graph_builder = es::EsGraphBuilder("pattern0");
    auto [a0, b0, c0] = graph_builder.CreateInputs<3>();   // 三个外部输入占位符
    auto matmul0 = es::MatMul(a0, b0);
    auto add0    = es::Add(matmul0, c0);
    auto graph0  = graph_builder.BuildAndReset({add0});    // add0 为 pattern 输出
    patterns.emplace_back(std::make_unique<Pattern>(std::move(*graph0)));
    return patterns;
  }

  // 2) 定义替换结构：GEMM(a, b, c, alpha, beta)
  GraphUniqPtr Replacement(const std::unique_ptr<MatchResult> &match_result) override {
    auto rb = es::EsGraphBuilder("replacement");
    auto [r_a, r_b, r_c] = rb.CreateInputs<3>();
    auto alpha = rb.CreateScalar(1);
    auto beta  = rb.CreateScalar(1);
    auto gemm  = es::GEMM(r_a, r_b, r_c, alpha, beta);
    return rb.BuildAndReset({gemm});
  }
};
```

### 4.3 注册

```cpp
// 注册到 InferShape 前阶段（最常用）
REG_FUSION_PASS(FuseMatMulAndAddPass).Stage(CustomPassStage::kBeforeInferShape);
```

> `MeetRequirements` 这里未重写，默认所有命中都替换。若只想在特定条件下融合（如 dtype/shape 满足要求），可重写该函数返回 true/false。

### 4.4 动手实践：编译并执行真实 C++ Fusion Pass

本教程以 CANN 9.0.0 为环境基线，下面使用该版本支持的 **C++ `PatternFusionPass`** 实现 `Add(x, 0) → x`。单元会在临时目录中自动生成 ES API、编译 `.so`，并在 `ge_initialize` 前加载完成注册；不写入 CANN 安装目录，也不需要 root 权限。

Pass 的 `Replacement` 会同时打印 `[PASS]` 日志并写入标记文件，随后融合后的图会在 0 号 NPU 上执行并与输入对拍。只有“编译成功 + Replacement 真实执行 + NPU 数值正确”同时满足才算通过。

> **耗时提示**：首次需要生成 ES API、编译 C++ Pass 并在线编图，通常是本章较慢的样例，可能需要数分钟；运行时会显示 1/3～3/3 的当前步骤，其中第 1 步通常最久。


In [ ]:
# === 真机运行：编译 C++ Fusion Pass -> GE 编译期改图 -> NPU 对拍 ===
import os
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

source_dir = Path("Sources/03.06/add_zero_pass").resolve()
required_files = ["CMakeLists.txt", "src/add_zero_pass.cpp", "run_add_zero_pass.py", "run.sh"]
missing_files = [name for name in required_files if not (source_dir / name).is_file()]
if missing_files:
    raise FileNotFoundError("3.06 样例文件缺失：{}".format(", ".join(missing_files)))

# 在隔离目录中构建，避免不同运行共享 CMakeCache。
work_root = Path(tempfile.mkdtemp(prefix="ge_0306_"))
sample_dir = work_root / "add_zero_pass"
shutil.copytree(
    source_dir,
    sample_dir,
    ignore=shutil.ignore_patterns("build", "__pycache__", "*.pyc", "*.executed"),
)
print("样例隔离执行目录：", sample_dir)

env = os.environ.copy()
env["PYTHON_EXECUTABLE"] = sys.executable
process = subprocess.Popen(
    ["bash", "run.sh"],
    cwd=sample_dir,
    env=env,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    bufsize=1,
)
log_lines = []
for line in process.stdout:
    log_lines.append(line)
    if line.startswith(("[INFO]", "[PASS]", "[OK]", "[ERROR]")):
        print(line.rstrip())
return_code = process.wait()
if return_code != 0:
    print("[ERROR] 流水线末尾日志：")
    print("".join(log_lines[-80:]))
    raise RuntimeError("C++ Fusion Pass 真机验证失败，退出码={}".format(return_code))

pass_library = sample_dir / "build/es_output/lib64/libnotebook_add_zero_pass.so"
marker_file = sample_dir / "notebook_add_zero_pass.executed"
assert pass_library.is_file(), "Fusion Pass 动态库未生成"
assert marker_file.read_text(encoding="utf-8").strip() == "NotebookAddZeroPass executed"
print("[OK] 3.06 一键验证完成")


## 5. Pattern 的边界规则（最易错点）

Pattern 边界是写 Pattern Pass 时最容易出错的部分。记住一个原则：

> **Pattern 要完整说明这段子图从哪里接收外部输入，以及哪些输出在替换后还要交给外部使用。**

### 5.1 输入边界

凡是来自匹配子图**外部**的 Tensor，都要在 pattern 里用输入占位符表示；pattern 内部创建的常量（如 `Add(x, 0)` 里的 `0`）不算外部输入。

### 5.2 输出边界 & 自包含约束

凡是替换后还要被子图外部使用的 Tensor，**必须**声明为 pattern 的输出。pattern 内部普通节点的某个输出若没声明为 pattern 输出，则它的所有消费者必须都在 pattern 内部（自包含）。否则替换后会出现「外部还想用这个 Tensor，但 replacement 没提供输出口」，图会断开。

```
X ---- A ---- out0
|
out1            ← 如果 X 的输出也被 pattern 外部使用，必须把 out1 也声明为 pattern 输出
```

自检三问：
1. 这个 Tensor 替换后还会被外部节点使用吗？
2. 如果会，它是否已作为 pattern 输出声明？
3. 如果不会，它的消费者是否都在 pattern 内部？

### 5.3 输入个数要精确

普通算子节点的输入个数必须与真实图一致。真实图某算子有 3 个输入，pattern 对应节点也要有 3 个输入——即使某些输入不关心来源，也要用占位符补齐。

### 5.4 不支持的 Pattern 内容

| 内容 | 原因 |
| --- | --- |
| 控制边 | matcher 不按控制依赖匹配，仅匹配数据边 |
| 子图 | 不支持嵌套子图，Pattern 必须是扁平 DAG |
| 动态输入/输出个数的节点 | 边界不确定，匹配歧义 |

### 5.5 多输出 Pattern

一个 pattern 可以有多个输出（一次匹配暴露多个输出 Tensor）。但如果只是想支持多种拓扑（如同时支持 `MatMul+Add` 和 `BatchMatMulV2+Add`），应**定义多个 pattern**，而不是用多输出。

## 6. 常用扩展点

### 6.1 CaptureTensor —— 读取中间 Tensor

当 `MeetRequirements` 或 `Replacement` 需要读取 pattern 中某个中间 Tensor 对应的真实节点（如读 MatMul 输出的 dtype/shape，或读原节点属性写到新节点）时，可在定义 pattern 时捕获它，匹配成功后从 `MatchResult` 按捕获顺序取回。

```cpp
// 捕获 relu 的生产者输出
pattern->CaptureTensor({*relu.GetProducer(), 0});

// MeetRequirements 中取回并校验（示例：要求 ReLU 输出为动态 shape）
NodeIo relu_output;
if (match_result->GetCapturedTensor(0, relu_output) != GRAPH_SUCCESS) return false;
TensorDesc desc;
relu_output.node.GetOutputDesc(relu_output.index, desc);
return desc.GetShape().GetShapeSize() == -1;
```

### 6.2 PatternMatcherConfig —— 更严格的匹配

默认 matcher 主要匹配拓扑与算子类型。若希望更严格，可启用配置：

| 配置 | 作用 |
| --- | --- |
| `EnableConstValueMatch` | 匹配 Const 值，值相等才算匹配成功（如要求 `Add(x, 0)` 的常量必须是 0） |
| `EnableIrAttrMatch` | 匹配 IR 属性的数量和值 |

```cpp
// 在自定义 Pass 构造函数中打开 Const 值匹配
explicit CustomFusionPass()
    : PatternFusionPass(PatternMatcherConfigBuilder().EnableConstValueMatch().Build()) {}
```

> 经验法则：判断逻辑简单、严格、稳定时放进 matcher 配置；需要容差、dtype 归一化或多条件组合时，放进 `MeetRequirements` 更清晰。

### 6.3 DecomposePass —— 单算子展开为多算子

`DecomposePass` 不需要先定义 pattern 图，而是在构造时直接声明要处理的算子类型。`Replacement(const GNode &)` 是纯虚函数，必须重写；`MeetRequirements(const GNode &)` 有默认实现，只有需要按属性过滤时才重写。典型用途是把 `groups > 1` 的 `Conv2D` 拆成 `Split + Conv2D*N + Concat`。

下面用 `Swish(x) → x * Sigmoid(x)` 展示一个最小、可实例化的 1→N Pass：

```cpp
#include "ge/fusion/pass/decompose_pass.h"
#include "es_all_ops.h"

using namespace ge;
using namespace ge::fusion;

class DecomposeSwishPass : public DecomposePass {
 public:
  explicit DecomposeSwishPass(const std::vector<AscendString> &op_types)
      : DecomposePass(op_types) {}

 protected:
  GraphUniqPtr Replacement(const GNode &) override {
    auto builder = es::EsGraphBuilder("replacement");
    auto [x] = builder.CreateInputs<1>();
    auto sigmoid = es::Sigmoid(x);
    auto output = es::Mul(x, sigmoid);
    return builder.BuildAndReset({output});
  }
};

REG_DECOMPOSE_PASS(DecomposeSwishPass, {"Swish"})
    .Stage(CustomPassStage::kBeforeInferShape);
```

若只希望处理满足特定条件的节点，可再重写 `MeetRequirements`。CANN 9.1.0 还提供 `DecomposePassV2`，其两个钩子会额外接收 `CustomPassContext`，注册方式与 V1 相同。

## 7. Pass 执行阶段（Stage）

普通融合 Pass 注册时要指定执行阶段。阶段决定 Pass 能看到的图状态，也决定 Replacement 是否需要自行做 shape 推导。

| 阶段 | C++ 枚举 | 使用建议 |
| --- | --- | --- |
| InferShape 前 | `CustomPassStage::kBeforeInferShape` | **最常用**。replacement 后续会进入统一 shape 推导流程 |
| InferShape 后 | `CustomPassStage::kAfterInferShape` | replacement 需自行保证输出 shape 等信息正确 |
| 内置融合后 | `CustomPassStage::kAfterBuiltinFusionPass` | 希望在 GE 内置融合完成后再处理时使用 |
| 原图优化后 | `CustomPassStage::kAfterOriginGraphOptimize` | 希望在原图优化结束后追加自定义处理时使用 |

> `CustomPassStage::kAfterAssignLogicStream` 是自定义逻辑流分配 Pass 的专用阶段，只支持 `CustomAllocateStreamPassFn`，不属于普通融合 Pass 的可选阶段；普通融合 Pass 误注册到该阶段会被忽略。流分配 Pass 通过专用接口注册，无需调用 `.Stage(...)`，详见第 11 节。

> 初次开发优先选 **InferShape 前**。只有当判断必须依赖已推导完成的 shape，或 replacement 本身会调用 shape 推导时，才用 InferShape 后阶段；此时若用到 `GeUtils::CheckNodeSupportOnAicore` 等接口，需在 InferShape 之后调用。

不同阶段对应不同的 dump 图文件名（用于验证，见第 10 节）：

- InferShape 前：`ge_onnx_xxxx_PreRunBegin.pbtxt`（融合前）→ `ge_onnx_xxxx_RunCustomPassBeforeInfershape.pbtxt`（融合后）
- InferShape 后：`..._PrepareAfterInferFormatAndShape.pbtxt` → `..._RunCustomPass_AfterInferShape.pbtxt`
- 内置融合后：`..._OptimizeOriginalGraph_FeGraphFusionAfter.pbtxt` → `..._RunCustomPassAfterBuiltinFusionPass.pbtxt`

## 8. 用改图接口实现 Pass

当 Pattern 无法表达你的变换，或需要完全手动控制图结构时，用改图接口路线：编写一个改图函数并用 `REGISTER_CUSTOM_PASS` 注册。

```cpp
#include "register_custom_pass.h"
// 自定义改图函数
graphStatus CustomPassFunc(GraphPtr &graph, CustomPassContext &custom_context) {
    // 在此定义图修改具体行为
    return GRAPH_SUCCESS;  // 成功
}
REGISTER_CUSTOM_PASS("FuseMatMulAndAddPass")
    .CustomPassFn(CustomPassFunc)
    .Stage(CustomPassStage::kBeforeInferShape);
```

> `graphStatus` / `Status` 是无符号状态类型。失败时应返回框架定义的状态码（如 `GRAPH_FAILED`、`GRAPH_PARAM_INVALID`），不要自行使用所谓“小于 0 的错误码”；需要补充诊断信息时，可先调用 `custom_context.SetErrorMessage("...")`。

改图函数内部的典型「五步走」（以 MatMul+Add → GEMM 为例）：

```
1. FindNodes        遍历 graph->GetAllNodes() 找到 MatMul、Add 节点
2. CheckNodesHaveEdge  确认 MatMul 输出直连 Add
3. CreateGEMMNode   创建 GEMM + alpha/beta 常量，graph->AddNodeByOp / AddDataEdge
4. AddInputsAndOutputs  把 a/b/c 接到 GEMM，UpdateInputDesc/UpdateOutputDesc
5. RemoveOldNodes   RemoveEdge 删旧边、把输出改接到 GEMM、RemoveNode 删旧节点
```

关键接口片段：

```cpp
auto gemm = op::GEMM(kOpNameGEMM);
auto node_gemm = graph->AddNodeByOp(gemm);              // 加节点
graph->AddDataEdge(*a, a_output_index, node_gemm, 0);   // 建数据边 a → GEMM:0
graph->RemoveEdge(dst_node, 0, *out_node, out_id);      // 删旧边
graph->RemoveNode(node);                                // 删旧节点
```

> 改图时只能使用 `CustomPassContext`、`Graph`、`GNode`、`PassReceiver`、`PassRegistrationData`、`REGISTER_CUSTOM_PASS`、`StreamPassContext` 等开放接口。改图接口需判断返回值；若 replacement 要用到 CANN 不支持的算子，需先用 Ascend C 自定义该算子（见上一节）。

## 9. 编译部署

无论哪条路线，Pass 都要编译为 `.so` 插件并安装到指定目录，才能被 GE 在图编译时加载。

### 9.1 编译为 .so

```bash
source ${ASCEND_PATH}/set_env.sh   # 配置 CANN 环境变量
mkdir build && cd build
cmake ..
make -j$(nproc) fuse_matmul_add_pass
make install                       # 安装 .so 到自定义融合 Pass 目录
```

> CMakeLists 中按需修改 `ASCEND_PATH`、`PASS_SO_DIR`、`target_include_directories`、`target_link_libraries`。注意**禁止链接软件包中的其他 so**，否则后续升级可能导致兼容性问题；若网络中有自定义算子，需增加其原型定义头文件。

### 9.2 安装路径

`.so` 需放到（或软链接到）如下目录，GE 会自动扫描加载：

```
${INSTALL_DIR}/opp/vendors/<xxx>/custom_fusion_passes/
                          └─ 仅一层自定义目录    └─ 该目录下不能有子目录
```

- `${INSTALL_DIR}`：CANN 安装路径（root 安装默认 `/usr/local/Ascend/cann`）。
- 多个 `vendors/<xxx>` 目录按字母序遍历，单个目录内 `.so` 按字母序加载，非 `.so` 文件跳过。
- `.so` 对执行用户需有可读权限。

## 10. 融合验证

验证 Pass 是否生效，需要从**结构生效**和**语义正确**两个层面确认。

### 10.1 dump 图对比（结构生效）

```bash
export DUMP_GE_GRAPH=1     # 编译模型前设置，dump 出编译过程中的图
# 然后用任一入口编译模型：ATC 工具 / 编译 Graph 为离线模型 / 编译并运行 Graph
atc --model=./model.onnx --framework=5 --soc_version=xxx --output=./model
```

执行完会生成一系列 `ge_onnx_*.pbtxt`。以 InferShape 前阶段为例，对比：

- `ge_onnx_xxxx_PreRunBegin.pbtxt`（融合前）
- `ge_onnx_xxxx_RunCustomPassBeforeInfershape.pbtxt`（融合后）

用 Netron 等可视化软件打开，确认 `MatMul + Add` 已替换为单个 `GEMM`，即说明 Pass **在结构上生效**。

### 10.2 数值对拍（语义正确）

dump 图只能确认拓扑变换已发生，**不能**证明变换后的语义与原图等价。必须对融合前后的模型用相同输入执行，比对输出是否一致：

- 融合前：用原始模型（或关闭自定义 Pass 后编译的模型）跑一组输入，记录输出。
- 融合后：用融合后的模型跑相同输入，比对输出。
- 比对方式：逐元素计算最大绝对误差 / 相对误差，在可接受阈值内才算通过。

> 本节 4.4 动手实践中的真机验证已经体现了这一原则——只有"编译成功 + Replacement 真实执行 + NPU 数值正确"同时满足才算通过。

### 10.3 日志确认

在 Pass 中加入打印（如示例里的 `Define pattern for FuseMatMulAndAddPass`），运行时若日志出现这些打印，说明 Pass 被调用。定位问题时可让日志打印到屏幕并调到 debug 级别：

```bash
export ASCEND_SLOG_PRINT_TO_STDOUT=1   # 日志打印到屏幕
export ASCEND_GLOBAL_LOG_LEVEL=0       # debug 级别
# ATC 场景还可加 --log=debug
```

### 10.4 收益确认

结构生效且语义正确后，是否真有性能收益还需结合 Profiling 看融合后算子的实际耗时（融合减少了算子数量和中间 Tensor 读写）。

## 11. 进阶：自定义逻辑流分配 Pass 定制并发

除了改图融合，GE 还开放了一类特殊的自定义 Pass：**自定义逻辑流分配 Pass**，用于调整算子在逻辑流上的分配，从而定制并发。

动机：算子执行时间、核占用、带宽等因素使同一算法在不同硬件上并发效果不同，编译期难以完美预估。该特性允许结合图拓扑和 Profiling 分析，灵活调整并发，做到「一网一策、一卡一策」。两种典型用法：基于内置流分配结果微调，或基于图结构开发自研流分配算法。

```cpp
#include "register_custom_pass.h"
// 自定义流分配 Pass：给指定节点分配新的 Stream ID 以触发并发
graphStatus AllocateStreamPass(const ConstGraphPtr &graph, StreamPassContext &context) {
    for (const auto &node : graph->GetDirectNode()) {  // 只遍历当前图的直接节点（不递归子图）
        AscendString node_name;
        node.GetName(node_name);
        if (std::string(node_name.GetString()) == "Abs_1/...") {
            context.SetStreamId(node, context.AllocateNextStreamId());  // 分配新流
        }
    }
    return GRAPH_SUCCESS;
}
// 注册：无需指定 Stage，默认在逻辑流分配阶段之后执行
REGISTER_CUSTOM_PASS("AllocateStreamPass").CustomAllocateStreamPassFn(AllocateStreamPass);
```

要点：
- **只读约束**——回调接收 `const ConstGraphPtr &`，只能读取图结构，不能增删节点或边；流 ID 必须通过 `StreamPassContext` 调整。
- 概念区分——**逻辑流**是按拓扑序/引擎归属预分配的流，同一逻辑流上的任务按序执行；**流分配**指定哪些任务可并行。
- 开启强制单流开关（`ge.enableSingleStream=true`）时，自定义逻辑流分配 Pass 不会被执行。
- 验证：对比 `..._RunCustomPass_BeforeAssignLogicStream*.pbtxt` 与 `..._RunCustomPass_AfterAssignLogicStream*.pbtxt` 看节点的 Stream ID 变化；并发是否真有收益需结合 Profiling 判断（有并发条件的算子若抢占同一计算资源不一定有收益）。

## 小结

- 自定义融合 Pass 在图编译期识别一段结构并替换为等价、更高效的结构，由 GE 在指定阶段自动调用，**不改 GE 框架代码**，以 `.so` 插件安装到 `opp/vendors/<xxx>/custom_fusion_passes/`。
- 两条实现路线：**Pattern 匹配**（推荐，继承 `PatternFusionPass`/`DecomposePass`，三步：匹配→决策→替换）与**改图接口**（手动遍历改图，控制力最强）。
- PatternFusionPass 三个核心函数：`Patterns`（必写）、`MeetRequirements`（可选过滤）、`Replacement`（必写）；用 `REG_FUSION_PASS` 注册并指定 `Stage`。
- **Pattern 边界规则是最易错点**：外部输入要声明、外部还要用的输出要声明、输入个数要精确、不支持控制边/子图/动态输入个数。
- 扩展点：`CaptureTensor`（读中间 Tensor）、`PatternMatcherConfig`（Const 值/IR 属性匹配）、`DecomposePass`（单算子展开）。
- 验证靠 `DUMP_GE_GRAPH=1` 对比融合前后 dump 图 + 日志打印 + Profiling 看收益。
- 进阶：自定义逻辑流分配 Pass 可结合 Profiling 定制并发。

至此「图编译扩展能力」两节学习完毕。下一节进入**章节练习**，综合检验 3.2–3.6 的内容。

## 课后练习

本节介绍了自定义融合 Pass 的实现路线、执行流程与验证方法，请完成以下题目自测。

1. （判断题）自定义融合 Pass 的目标是在图中找到一段符合规则的结构，替换为等价但更高效的结构，且不改变模型语义。

2. （判断题）基于 Pattern 匹配实现的融合 Pass 支持在 Pattern 中使用控制边和嵌套子图进行匹配。

3. （单选题）`PatternFusionPass` 中**必须**重写、用于定义匹配模板的函数是哪个？
    A. `MeetRequirements`
    B. `Patterns`
    C. `Run`
    D. `Serialize`

4. （单选题）自定义融合 Pass 初次开发时，最常用、且 replacement 后续会进入统一 shape 推导流程的执行阶段是？
    A. `kAfterInferShape`
    B. `kBeforeInferShape`
    C. `kAfterBuiltinFusionPass`
    D. `kAfterOriginGraphOptimize`

5. （单选题）要把一个 `groups > 1` 的 `Conv2D` 拆成 `Split + Conv2D*N + Concat`（单算子展开为多算子），最适合用哪个基类？
    A. `PatternFusionPass`
    B. `DecomposePass`
    C. `EagerExecuteOp`
    D. `CompilableOp`

6. （多选题）以下关于 Pattern 边界规则的描述，哪些正确？
    A. 来自匹配子图外部的 Tensor 必须用输入占位符声明
    B. 替换后仍被子图外部使用的输出必须声明为 pattern 输出
    C. pattern 中普通算子节点的输入个数必须与真实图一致
    D. pattern 内部创建的常量也必须声明为外部输入

7. （多选题）以下关于自定义融合 Pass 编译部署与验证的描述，哪些正确？
    A. Pass `.so` 需放到 `opp/vendors/<xxx>/custom_fusion_passes/` 目录被 GE 加载
    B. 设置 `DUMP_GE_GRAPH=1` 后可对比融合前后的 dump 图确认 Pass 生效
    C. 编译 Pass 时建议链接软件包中尽可能多的 so 以保证兼容性
    D. 自定义逻辑流分配 Pass 在开启强制单流开关时不会被执行

**执行以下代码获取答案。**

In [ ]:
!cat ./answer/03.06_answer.txt